# 25 — Policy gradient: REINFORCE and the PPO clip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/025-policy-gradient-at-scale.ipynb)

Instead of learning values first, policy-gradient methods turn the policy itself into the thing you nudge: good outcomes make the action taken more likely, bad ones make it rarer. You will run REINFORCE on a two-armed bandit, watch the policy move, and then look at the clip that PPO puts on the size of an update.

## 1. The numbers: one parameter, two arms

The policy is a single number `p`, the probability of pulling arm 1, and each arm pays 1 with a fixed chance. With `p = 0.5` the expected reward is the plain average of the two arms. The last line shows how the two log-probability gradients cancel, because probabilities must sum to one.

In [ ]:
import numpy as np                                   # the array library
p = 0.5                                              # the current probability of pulling arm 1
means = np.array([0.35, 0.65])                       # true chance of a reward on each arm
expected = np.array([1 - p, p]) @ means              # value of the current policy
print("expected reward:", round(expected, 3))        # 0.5 * 0.35 + 0.5 * 0.65
print("best arm pays:", means.max())                 # 0.65
grad_log = np.array([-p, 1 - p])                     # d log pi / d theta for each action
print("grad of log pi:", grad_log)                   # arm 1 gets +0.5, arm 0 gets -0.5
print("they sum to:", grad_log.sum())                # 0.0: the two gradients cancel

**Line by line**

- `import numpy as np` — loads the array library.
- `p = 0.5` — the single number that is the whole policy.
- `means = ...` — the true reward rate of each arm.
- `expected = ...` — probability of each action times its reward rate.
- `print("expected reward:", ...)` — what the policy is worth right now.
- `print("best arm pays:", ...)` — the ceiling.
- `grad_log = ...` — how the log-probability of each action moves with the parameter.
- `print("grad of log pi:", ...)` — a push toward one arm is a pull away from the other.
- `print("they sum to:", ...)` — the gradients cancel.

## 2. Step by step: REINFORCE, one episode at a time

Each episode pulls one arm, sees a 0 or a 1, and compares that reward with a running average. The surprise, reward minus baseline, scales the push toward whatever arm was just pulled, so a lucky pull makes that arm more likely and an unlucky one makes it rarer.

In [ ]:
rng = np.random.default_rng(0)                       # fixed seed: the run is reproducible
theta = 0.0                                          # the parameter; p = sigmoid(theta)
baseline = 0.0                                       # running average reward
p_hist = []                                          # probability of arm 1 after each episode
for ep in range(400):                                # 400 episodes
    p = 1. / (1. + np.exp(-theta))                   # sigmoid: parameter to probability
    a = int(rng.random() > p)                        # pull arm 1 with probability p
    reward = float(rng.random() < means[a])          # a Bernoulli reward from that arm
    advantage = reward - baseline                    # better or worse than usual?
    theta = theta + 0.3 * advantage * (a - p)        # push toward the action just taken
    baseline = baseline + 0.05 * (reward - baseline) # track the average reward
    p_hist.append(1. / (1. + np.exp(-theta)))        # record the policy after the update
    if ep % 100 == 0:                                # four checkpoints
        print(f"episode {ep:3d}  p(arm 1) {p_hist[-1]:.3f}  baseline {baseline:.3f}")   # progress

**Line by line**

- `rng = np.random.default_rng(0)` — a seeded random source.
- `theta = 0.0` — the parameter behind the policy.
- `baseline = 0.0` — the running average that defines surprise.
- `p_hist = []` — the policy's history.
- `for ep in range(400):` — the episode loop.
- `p = 1. / (1. + np.exp(-theta))` — sigmoid turns any number into a probability.
- `a = int(rng.random() > p)` — sample an action.
- `reward = float(...)` — sample the reward.
- `advantage = reward - baseline` — the surprise.
- `theta = theta + 0.3 * ...` — REINFORCE: step size times surprise times gradient.
- `baseline = ...` — a slow average of rewards.
- `p_hist.append(...)` — record the new policy.
- `if ep % 100 == 0:` — print at four checkpoints only.
- `print(...)` — policy and baseline at each checkpoint.

## 3. Plot: the policy climbing

Single episodes are noisy, so the light line is the raw policy and the dark one is a 25-episode average. The policy rises toward always-arming 1, but it approaches the top asymptotically: each step is proportional to how uncertain the policy still is.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
smooth = np.convolve(p_hist, np.ones(25) / 25, mode="valid")   # 25-episode moving average
plt.figure(figsize=(6, 3))                           # one wide panel
plt.plot(p_hist, color="tab:blue", alpha=0.35, label="p(arm 1) each episode")   # raw policy
plt.plot(range(24, len(p_hist)), smooth, color="tab:red", label="25-episode average")   # smoothed
plt.axhline(1.0, color="grey", linestyle="--", label="always pull arm 1")   # the limit
plt.ylim(0, 1.05)                                    # probability axis
plt.xlabel("episode")                                # x-axis label
plt.ylabel("probability of pulling arm 1")           # y-axis label
plt.legend(fontsize=8)                               # name the three lines
plt.title("the policy finds the arm worth 0.65 and closes in slowly")   # takeaway
plt.show()                                           # render

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `smooth = np.convolve(...)` — a 25-episode moving average of the policy.
- `plt.figure(...)` — one panel.
- `plt.plot(p_hist, ...)` — every episode's policy, faintly.
- `plt.plot(range(24, ...), smooth, ...)` — the average, aligned with its episodes.
- `plt.axhline(1.0, ...)` — the policy the learner is heading toward.
- `plt.ylim(0, 1.05)` — a probability axis.
- `plt.xlabel("episode")` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend(...)` — name the lines.
- `plt.title(...)` — the sentence the figure is making.
- `plt.show()` — render.

## 4. Interactive: drag the step size

The slider sets the learning rate and redraws the policy over the same 400 episodes with the same seed, so only the step size changes. Small steps leave the policy near 0.5; large steps race to the top. The title prints the policy at the last episode.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function
def show(lr=0.1):                                    # lr comes from the slider
    rng = np.random.default_rng(0)                   # same seed every drag
    theta, base, hist = 0.0, 0.0, []                 # parameter, baseline, history
    for ep in range(400):                            # the same 400 episodes
        p = 1. / (1. + np.exp(-theta))               # current policy
        a = int(rng.random() > p)                    # sample an action
        reward = float(rng.random() < means[a])      # sample a reward
        theta = theta + lr * (reward - base) * (a - p)   # REINFORCE update, sized by the slider
        base = base + 0.05 * (reward - base)         # update the baseline
        hist.append(1. / (1. + np.exp(-theta)))      # record the policy
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(hist, color="tab:green")                # policy against episode
    plt.ylim(0, 1.05)                                # fixed axis, so curves are comparable
    plt.xlabel("episode")                            # x-axis label
    plt.ylabel("p(arm 1)")                           # y-axis label
    plt.title(f"lr = {lr:.2f}: final p = {hist[-1]:.2f}")   # live readout
    plt.show()                                       # draw it
    print(f"lr {lr:.2f} -> final p {hist[-1]:.3f}")  # the same number as text
interact(show, lr=(0.01, 0.5, 0.01))                 # the slider

**Line by line**

- `from ipywidgets import interact` — the slider widget.
- `def show(lr=0.1):` — one function per slider position.
- `rng = np.random.default_rng(0)` — the same random draws for every setting.
- `theta, base, hist = ...` — start clean.
- `for ep in range(400):` — the episode loop.
- `p = ...` — current policy.
- `a = ...` — sample an action.
- `reward = ...` — sample a reward.
- `theta = theta + lr * ...` — the update, sized by the slider.
- `base = ...` — update the baseline.
- `hist.append(...)` — record the policy.
- `plt.figure(...)` — a new figure per drag.
- `plt.plot(hist, ...)` — the policy curve.
- `plt.ylim(0, 1.05)` — a fixed axis for comparison.
- `plt.xlabel("episode")` — names the horizontal axis.
- `plt.ylabel("p(arm 1)")` — names the vertical axis.
- `plt.title(...)` — the final policy, printed on the plot.
- `plt.show()` — draw it.
- `print(...)` — the same number as text.
- `interact(show, lr=(0.01, 0.5, 0.01))` — the slider, its range and its step.

## 5. When it breaks: the clip is silent inside the band

PPO multiplies the advantage by the ratio of new to old action probability, then clamps that ratio to the band 0.7 to 1.3. A ratio of 1.2 sits inside the band, so the objective pays the full 1.2*A and nothing is protected. Past 1.3 the objective is flat, so the gradient is exactly zero and the step is never undone.

In [ ]:
ratio = np.linspace(0.4, 2.0, 161)                   # how much the new policy grew: new / old
clip = np.clip(ratio, 0.7, 1.3)                      # PPO's trusted band
A = 1.0                                              # a positive advantage: this action paid off
objective = np.minimum(ratio * A, clip * A)          # the clipped surrogate for A > 0
neg = np.minimum(ratio * -A, clip * -A)              # the same rule for A < 0
print("ratio 1.2, A = +1 -> objective", np.minimum(1.2, 1.3) * A)   # 1.2: the clip never binds
print("ratio 1.5, A = +1 -> objective", np.minimum(1.5, 1.3) * A)   # 1.3: clamped, and flat after
plt.figure(figsize=(6, 3))                           # one panel
plt.plot(ratio, objective, label="A = +1")           # positive advantage
plt.plot(ratio, neg, label="A = -1")                 # negative advantage
plt.axvline(0.7, color="grey", linestyle="--")       # lower edge of the band
plt.axvline(1.3, color="grey", linestyle="--")       # upper edge of the band
plt.axhline(1.3, color="tab:red", linestyle=":", linewidth=1)   # where A = +1 stops paying
plt.plot([1.2], [1.2], "ko")                         # inside the band: paid in full
plt.plot([1.5], [1.3], "ks")                         # outside the band: frozen at 1.3
plt.xlabel("policy ratio, new / old")                # x-axis label
plt.ylabel("objective, ratio x advantage")           # y-axis label
plt.legend(fontsize=8)                               # name the two curves
plt.title("inside the band the clip is silent; outside it the slope is zero")   # takeaway
plt.show()                                           # render

**Line by line**

- `ratio = np.linspace(0.4, 2.0, 161)` — candidate ratios from shrunken to doubled.
- `clip = np.clip(ratio, 0.7, 1.3)` — the ratio after PPO's clamp.
- `A = 1.0` — the advantage, held positive.
- `objective = ...` — the smaller of clipped and unclipped, for A > 0.
- `neg = ...` — the same formula with the sign flipped.
- `print("ratio 1.2 ...")` — 1.2 * A: the clip did nothing.
- `print("ratio 1.5 ...")` — 1.3 * A: clamped, and no longer changing.
- `plt.figure(...)` — one panel.
- `plt.plot(ratio, objective, ...)` — the positive-advantage objective.
- `plt.plot(ratio, neg, ...)` — the negative-advantage objective.
- `plt.axvline(0.7, ...)` — lower edge of the band.
- `plt.axvline(1.3, ...)` — upper edge.
- `plt.axhline(1.3, ...)` — the plateau for a positive advantage.
- `plt.plot([1.2], [1.2], "ko")` — the unclipped point.
- `plt.plot([1.5], [1.3], "ks")` — the clamped point.
- `plt.xlabel(...)` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend(...)` — name the curves.
- `plt.title(...)` — the sentence the figure is making.
- `plt.show()` — render.

## Your turn

Set `means` to `[0.45, 0.5]` and predict how many episodes the policy needs before it looks decided. Then compute by hand what the section 5 objective pays at a ratio of 0.6 with `A = +1`, and check your answer against the plot.